# Climate Risk Disclosure Scrapping
This webscrapping tool is developed to download data from the Climate Reporting Entity (CRE) Search Hub. 

```
In recognition of the ongoing impact that climate change has on New Zealand the Government introduced a requirement for large entities to prepare and lodge annual climate-related disclosures (climate statements). These entities are called Climate Reporting Entities (CREs) under the Financial Markets Conduct Act 2013.  

The aim of requiring CREs to consider and report on climate-related risks and opportunities is to encourage a transition to a low-emissions future.

CREs are required to lodge climate statements or exemption notices with the Registrar of Financial Service Providers within 4 months after the balance date of the entity. CREs with a balance date of 31 December 2023 will be the first to file their climate statements which are due by 30 April 2024.

Which entities are Climate Reporting Entities (CREs)
Around 200 entities in New Zealand will be required to prepare and lodge climate statements or exemption notices. CREs are a subset of FMC reporting entities, and include: 
 - Large registered banks, credit unions, and building societies. Those with total assets of more than $1 billion. 
 - Large managers of registered investment schemes (other than restricted schemes). Those with greater than $1 billion in total assets under management. 
 - Large licensed insurers. Those with greater than $1 billion in total assets or annual gross premium income greater than $250 million. 
 - Large listed issuers of quoted equity securities or quoted debt securities. An equity issuer is large if the market price or fair value of all of its equity securities exceeds $60 million and a debt issuer is large if the face value of its quoted debt exceeds $60 million. Issuers listed on growth markets are excluded from the climate reporting entity definition. 
 
The thresholds for each entity are calculated as at the balance date of their 2 preceding accounting periods. These thresholds will be amended from time to time to reflect the movements in the consumers price index.
```
The CRE search hub is this link: https://crd-app.companiesoffice.govt.nz/dashboard/

```
From 1 January 2024 CREs are required to prepare climate statements and lodge them on this register. To begin with, the register only includes CREs with a reporting period up to June 2024. We'll add the remaining CREs in due course.

```




In [1]:
#!pip install "selenium>=4.18.1"
#!pip install beautifulsoup4
#!pip install webdriver_manager
#!pip install requests
#!pip install --upgrade selenium

In [ ]:
import selenium
selenium.__version__

'4.43.0'

In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import ast, re, time, os, requests
import pandas as pd
import numpy as np
from datetime import date

In [13]:
# ── Configuration ──────────────────────────────────────────────────────────────
BASE_URL      = 'https://crd-app.companiesoffice.govt.nz'
DASHBOARD_URL = f'{BASE_URL}/dashboard/'

# Output folder for CSVs and PDFs — update if running on a different machine
FOLDER     = r'C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure'
PDF_FOLDER = os.path.join(FOLDER, 'pdfs_2026')

os.makedirs(PDF_FOLDER, exist_ok=True)

# Reporting-period years to target for PDF download
TARGET_YEARS = [2024, 2025]

print(f"CSV output  : {FOLDER}")
print(f"PDF output  : {PDF_FOLDER}")
print(f"Target years: {TARGET_YEARS}")

CSV output  : C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure
PDF output  : C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure\pdfs_2026
Target years: [2024, 2025]


In [14]:
# Selenium ChromeDriver setup
CHROMEDRIVER_PATH = r"C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\4-Github2026_LLM\chromedriver-win64\chromedriver.exe"

options = webdriver.ChromeOptions()

# Use new headless mode
options.add_argument("--headless=new")
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")

# PDF download settings
prefs = {
    "download.default_directory": PDF_FOLDER,  # must be absolute path
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True,
    "safebrowsing.enabled": True,
}
options.add_experimental_option("prefs", prefs)

# Use your local ChromeDriver
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=options,
)

# Enable downloads in headless mode
driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": PDF_FOLDER,
    },
)

print("Chrome driver ready")

Chrome driver ready


The first step is to download the list of all entities 

In [6]:
# ── Step 1: Scrape the full entity list with dynamic pagination ─────────────────
driver.get(DASHBOARD_URL)
time.sleep(3)

# Detect total pages dynamically instead of hardcoding
try:
    page_btns = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'a[data-page]'))
    )
    total_pages = max(int(b.get_attribute('data-page')) for b in page_btns)
except Exception:
    total_pages = 25  # safe fallback
print(f"Pages to scrape: {total_pages}")

table_final = pd.DataFrame()

for page_num in range(1, total_pages + 1):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    try:
        btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, f'a[data-page="{page_num}"]'))
        )
        btn.click()

        table = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR,
                '#mainContent > div.entitylist > div > div.view-grid.table-responsive.has-pagination > table'))
        )
        table_html = table.get_attribute("outerHTML")
        soup = BeautifulSoup(table_html, 'html.parser')
        rows_html = soup.find_all('tr')

        table_data = []
        for row in rows_html:
            cells = row.find_all(['th', 'td'])
            row_data = [c.get_text(strip=True) for c in cells]
            # Collect entity-page hrefs from every cell in this row
            href_values = []
            for c in cells:
                a = c.find('a')
                if a and str(a.get('href', '')).startswith('/dashboard/reportingentity/?id='):
                    href_values.append(a['href'])
            row_data.append(list(set(href_values)))
            table_data.append(row_data)

        page_df = pd.DataFrame(table_data)
        table_final = pd.concat([table_final, page_df], ignore_index=True)
        print(f"  Page {page_num}/{total_pages} — {len(table_final)} entities so far")

    except Exception as e:
        print(f"  Page {page_num} error: {e}")
        continue

print(f"\nTotal rows scraped: {len(table_final)}")

Pages to scrape: 25
  Page 1/25 — 11 entities so far
  Page 2/25 — 22 entities so far
  Page 3/25 — 33 entities so far
  Page 4/25 — 44 entities so far
  Page 5/25 — 55 entities so far
  Page 6/25 — 66 entities so far
  Page 7/25 — 77 entities so far
  Page 8/25 — 88 entities so far
  Page 9/25 — 99 entities so far
  Page 10/25 — 110 entities so far
  Page 11/25 — 121 entities so far
  Page 12/25 — 132 entities so far
  Page 13/25 — 143 entities so far
  Page 14/25 — 154 entities so far
  Page 15/25 — 165 entities so far
  Page 16/25 — 176 entities so far
  Page 17/25 — 187 entities so far
  Page 18/25 — 193 entities so far
  Page 19/25 — 199 entities so far
  Page 20 error: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6707636a5+15b35]
	chromedriver!GetHandleVerifier [0x7ff670763710+15ba0]
	chromedriver!(No symbol) [0x7ff6704ad8dd]
	chromedriver!(No symbol) [0x7ff670506c2e]
	chromedriver!(No symbol) [0x7ff670506f3c]
	chromedriver!(No symbol) [0x7ff670558247]
	chromedrive

In [7]:
table_final.columns = ['CRE Name', 'Reporting Entity Status', 'NZBN', 'Actions', 'Link']
table_final.drop(index=0, inplace=True)
table_final.reset_index(drop=True, inplace=True)
table_final.head(5)

,CRE Name,Reporting Entity Status,NZBN,Actions,Link
0,AA INSURANCE LIMITED,Registered,9429040865966,View details,[/dashboard/reportingentity/?id=8b6b5ca5-8394-...
1,AFT PHARMACEUTICALS LIMITED,Registered,9429038010415,View details,[/dashboard/reportingentity/?id=a96b5ca5-8394-...
2,AIA NEW ZEALAND LIMITED,Registered,9429039580948,View details,[/dashboard/reportingentity/?id=8d6b5ca5-8394-...
3,AIG INSURANCE NEW ZEALAND LIMITED,Registered,9429031310048,View details,[/dashboard/reportingentity/?id=4fe2dfb9-81f4-...
4,AIR NEW ZEALAND LIMITED,Registered,9429040402543,View details,[/dashboard/reportingentity/?id=ab6b5ca5-8394-...


In [8]:
today = date.today().strftime("%d/%m/%Y")
table_final.to_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026.csv'), index=False)
print(f"Saved {len(table_final)} entities on {today}")

Saved 198 entities on 01/05/2026


In [21]:
table_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026','List_of_reporting_firms_2026.csv'))

links_final = pd.DataFrame()
for _, row in table_final.iterrows():
    try:
        entity_links = ast.literal_eval(row['Link'])
    except Exception:
        continue
    full_links = [BASE_URL + lnk for lnk in entity_links]
    df = pd.DataFrame({'CRE Name': [row['CRE Name']] * len(full_links), 'Link': full_links})
    links_final = pd.concat([links_final, df], ignore_index=True)

table_final  = table_final.drop(columns=['Link'])
links_final  = links_final.drop_duplicates(subset=['Link']).reset_index(drop=True)
links_final = pd.merge(table_final, links_final, on='CRE Name', how='left')
links_final = links_final[links_final['CRE Name']!= 'CRE Name. sort ascending']
links_final.to_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026_withlink.csv'), index=False)
print(f"Total entity links: {len(links_final)}")
links_final.head(5)

Total entity links: 180


,CRE Name,Reporting Entity Status,NZBN,Actions,Link
0,AA INSURANCE LIMITED,Registered,9429040865966,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
1,AFT PHARMACEUTICALS LIMITED,Registered,9429038010415,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
2,AIA NEW ZEALAND LIMITED,Registered,9429039580948,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
3,AIG INSURANCE NEW ZEALAND LIMITED,Registered,9429031310048,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
4,AIR NEW ZEALAND LIMITED,Registered,9429040402543,View details,https://crd-app.companiesoffice.govt.nz/dashbo...


### Step 2: Visit each entity page, extract reporting-period text and links

In [22]:
links_final = pd.read_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026_withlink.csv'))

table_details_final = pd.DataFrame()

for idx, (_, row) in enumerate(links_final.iterrows()):
    firm_name = row['CRE Name']
    if idx % 20 == 0:
        print(f"[{idx+1}/{len(links_final)}] {firm_name}")
    link = row['Link']

    driver.get(link)
    time.sleep(2)

    try:
        table = WebDriverWait(driver, 30).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
        )
        text = table.text
        table_html = table.get_attribute("outerHTML")

        # Also capture any per-period hrefs so we can navigate directly later
        soup = BeautifulSoup(table_html, 'html.parser')
        period_hrefs = []
        for tr in soup.find_all('tr')[1:]:
            tds = tr.find_all('td')
            if not tds:
                continue
            a = tds[0].find('a')
            period_hrefs.append(a['href'] if (a and a.get('href')) else None)

        row_data = pd.DataFrame([{
            'Link': link,
            'Company Name': firm_name,
            'Text': text,
            'PeriodHrefs': str(period_hrefs),
        }])
        table_details_final = pd.concat([table_details_final, row_data], ignore_index=True)

    except Exception as e:
        print(f"  Error for {firm_name}: {e}")

table_details_final.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'), index=False)
print(f"\nSaved {len(table_details_final)} entity detail rows")

[1/180] AA INSURANCE LIMITED
[21/180] AUSTRALIAN FOUNDATION INVESTMENT COMPANY LIMITED
[41/180] COMMONWEALTH BANK OF AUSTRALIA
[61/180] FUNDROCK NZ LIMITED
[81/180] KINGFISH LIMITED
[101/180] NAPIER PORT HOLDINGS LIMITED
[121/180] QBE INSURANCE (AUSTRALIA) LIMITED
[141/180] SOUTHERN CROSS MEDICAL CARE SOCIETY
[161/180] TOWER LIMITED

Saved 180 entity detail rows


### Correct the write up of the reporting period

In [24]:
table_details_final = pd.read_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'))

table_details_final['text_new'] = (
    table_details_final['Text']
    .str.replace('\n', '|')
    .str.replace('Reporting Period', '')
    .str.replace('. sort descending', '')
    .str.replace('. sort ascending', '')
    .str.replace('Scheme Status', '')
    .str.replace('Scheme', '')
    .str.replace('Reporting Entity', '')
    .str.replace('Status', '')
    .str.replace('Registered', '_')
    .str.replace('Actions', '')
)

table_details_final.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'), index=False)
table_details_final.head(5)


,Link,Company Name,Text,PeriodHrefs,text_new
0,https://crd-app.companiesoffice.govt.nz/dashbo...,AA INSURANCE LIMITED,Reporting Period\n. sort descending\nActions\n...,"['#', '/dashboard/reportingentity/scheme/repor...",|||30-06-2025 - Select to view climate stateme...
1,https://crd-app.companiesoffice.govt.nz/dashbo...,AFT PHARMACEUTICALS LIMITED,Reporting Period\n. sort descending\nActions\n...,"['#', '/dashboard/reportingentity/scheme/repor...",|||31-03-2024 - Select to view climate stateme...
2,https://crd-app.companiesoffice.govt.nz/dashbo...,AIA NEW ZEALAND LIMITED,Reporting Period\n. sort descending\nActions\n...,"['#', '/dashboard/reportingentity/scheme/repor...",|||31-12-2024 - Select to view climate stateme...
3,https://crd-app.companiesoffice.govt.nz/dashbo...,AIG INSURANCE NEW ZEALAND LIMITED,Reporting Period\n. sort descending\nActions\n...,"['#', '/dashboard/reportingentity/scheme/repor...",|||31-12-2025 - Select to view climate stateme...
4,https://crd-app.companiesoffice.govt.nz/dashbo...,AIR NEW ZEALAND LIMITED,Reporting Period\n. sort descending\nActions\n...,"['#', '/dashboard/reportingentity/scheme/repor...",|||30-06-2024 - Select to view climate stateme...


### Check on the one with schemes only

In [25]:
table_details_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'))

scheme_rows = table_details_final[table_details_final['Text'].str.contains('Scheme', case=False)]
print(f"Scheme entities: {scheme_rows.shape[0]}")

table_details_final_scheme = pd.DataFrame()

for idx, (_, row) in enumerate(scheme_rows.iterrows()):
    firm_name = row['Company Name']
    link = row['Link']   # already a full URL from links_final
    print(f"[{idx+1}/{len(scheme_rows)}] {firm_name}")

    driver.get(link)
    time.sleep(3)

    try:
        table = WebDriverWait(driver, 30).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
        )
        table_html = table.get_attribute("outerHTML")
        soup = BeautifulSoup(table_html, 'html.parser')
        rows_html = soup.find_all('tr')

        table_data = []
        for tr in rows_html[2:]:        # skip 2 header rows specific to scheme tables
            cells = tr.find_all('td')
            if not cells:               # skip empty rows
                continue
            row_data = [c.get_text(strip=True) for c in cells]
            # Last cell: relative href to the scheme's own entity page
            href_list = []
            a = cells[-1].find('a')
            if a and str(a.get('href', '')).startswith('/dashboard/reportingentity/'):
                href_list.append(a['href'])
            row_data.append(href_list)
            table_data.append(row_data)

        page_df = pd.DataFrame(table_data)
        table_details_final_scheme = pd.concat([table_details_final_scheme, page_df], ignore_index=True)

    except Exception as e:
        print(f"  Error: {e}")

table_details_final_scheme.columns = ['Scheme', 'Scheme Status', 'Company Name', 'Detail', 'Link']
table_details_final_scheme.dropna(subset=['Detail'], inplace=True)
table_details_final_scheme.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_scheme_sub-entities_2026.csv'), index=False)
print(f"\nSaved {len(table_details_final_scheme)} scheme sub-entities")

Scheme entities: 25
[1/25] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[2/25] ANZ INVESTMENT SERVICES (NEW ZEALAND) LIMITED
[3/25] ANZ NEW ZEALAND INVESTMENTS LIMITED
[4/25] ASB GROUP INVESTMENTS LIMITED
[5/25] BNZ INVESTMENT SERVICES LIMITED
[6/25] BOOSTER INVESTMENT MANAGEMENT LIMITED
[7/25] BT FUNDS MANAGEMENT (NZ) LIMITED
[8/25] CENTURIA FUNDS MANAGEMENT (NZ) LIMITED
[9/25] FIRST MORTGAGE MANAGERS LIMITED
[10/25] FISHER FUNDS MANAGEMENT LIMITED
[11/25] FISHER FUNDS WEALTH LIMITED
[12/25] FUNDROCK NZ LIMITED
[13/25] GENERATE INVESTMENT MANAGEMENT LIMITED
[14/25] GOODMAN PROPERTY SERVICES (NZ) LIMITED
[15/25] HARBOUR ASSET MANAGEMENT LIMITED
[16/25] KIWI WEALTH INVESTMENTS LIMITED PARTNERSHIP
[17/25] MEDICAL FUNDS MANAGEMENT LIMITED
[18/25] MERCER (N.Z.) LIMITED
[19/25] MILFORD FUNDS LIMITED
[20/25] NEW ZEALAND FUNDS MANAGEMENT LIMITED
[21/25] NIKKO ASSET MANAGEMENT NEW ZEALAND LIMITED
[22/25] NORTHWEST HEALTHCARE PROPERTIES MANAGEMENT LIMITED
[23/25] PIE FUNDS MANAGEMENT LIMITED
[24/2

### Check on the one with scheme number and start schaping 

In [38]:

links_scheme_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_scheme_sub-entities_2026.csv'))

# ── Scrape each scheme entity's reporting periods ─────────────────────────────
table_details_final_scheme_detail = pd.DataFrame()

for idx, (_, row) in enumerate(links_scheme_final.iterrows()):
    firm_name = row['Company Name']
    if idx % 20 == 0:
        print(f"[{idx+1}/{len(links_scheme_final)}] {firm_name}")
    link =     BASE_URL +  ast.literal_eval(row['Link'])[0]
    driver.get(link)
    time.sleep(2)

    try:
        table = WebDriverWait(driver, 30).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
        )
        text = table.text
        table_html = table.get_attribute("outerHTML")

        soup = BeautifulSoup(table_html, 'html.parser')
        period_hrefs = []
        for tr in soup.find_all('tr')[1:]:
            tds = tr.find_all('td')
            if not tds:
                continue
            a = tds[0].find('a')
            period_hrefs.append(a['href'] if (a and a.get('href')) else None)

        row_data = pd.DataFrame([{
            'Link': link, 'Company Name': firm_name,
            'Text': text, 'PeriodHrefs': str(period_hrefs),
        }])
        table_details_final_scheme_detail = pd.concat(
            [table_details_final_scheme_detail, row_data], ignore_index=True)

    except Exception as e:
        print(f"  Error: {e}")

table_details_final_scheme_detail.to_csv(
    os.path.join(FOLDER, 'pdfs_2026', 'table_details_final.scheme_detail.csv'), index=False)

[1/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[21/125] BOOSTER INVESTMENT MANAGEMENT LIMITED
[41/125] CENTURIA FUNDS MANAGEMENT (NZ) LIMITED
[61/125] FISHER FUNDS MANAGEMENT LIMITED
[81/125] FUNDROCK NZ LIMITED
[101/125] MERCER (N.Z.) LIMITED
[121/125] SMARTSHARES LIMITED


#### Link between the investment subschmes and the detail from the link


In [ ]:
links_scheme_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_scheme_sub-entities_2026.csv'))
table_details_final_scheme_detail = pd.read_csv(
    os.path.join(FOLDER, 'pdfs_2026', 'table_details_final.scheme_detail.csv'))

table_details_final_scheme_detail['text_new'] = (
    table_details_final_scheme_detail['Text']
    .str.replace('\n', '|')
    .str.replace('Reporting Period', '').str.replace('. sort descending', '')
    .str.replace('. sort ascending', '').str.replace('Scheme Status', '')
    .str.replace('Scheme', '').str.replace('Reporting Entity', '')
    .str.replace('Status', '').str.replace('Registered', '_').str.replace('Actions', '')
)

# Merge in the Scheme name from the sub-entities list
table_details_final_scheme_detail = table_details_final_scheme_detail.merge(
    links_scheme_final[['Link', 'Scheme']].rename(columns={'Scheme': 'Scheme Name'}),
    on='Link', how='left')

table_details_final_scheme_detail.to_csv(
    os.path.join(FOLDER, 'pdfs_2026', 'table_details_final.scheme_detail.new.csv'), index=False)
table_details_final_scheme_detail.head(5)


                                                Link  \
0  https://crd-app.companiesoffice.govt.nz/dashbo...   
1  https://crd-app.companiesoffice.govt.nz/dashbo...   
2  https://crd-app.companiesoffice.govt.nz/dashbo...   
3  https://crd-app.companiesoffice.govt.nz/dashbo...   
4  https://crd-app.companiesoffice.govt.nz/dashbo...   

                                Company Name  \
0  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
1  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
2  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
3  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
4  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   

                                                Text  \
0  Reporting Period\n. sort descending\nActions\n...   
1  Reporting Period\n. sort descending\nActions\n...   
2  Reporting Period\n. sort descending\nActions\n...   
3  Reporting Period\n. sort descending\nActions\n...   
4  Reporting Period\n. sort descending\nActions\n...   

                                     

In [ ]:
table_details_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'))
table_details_final_scheme_detail = pd.read_csv(
    os.path.join(FOLDER, 'pdfs_2026', 'table_details_final.scheme_detail.new.csv'))

# Combine non-scheme companies with scheme entities
table_details_final_noscheme = table_details_final[
    ~table_details_final['Text'].str.contains('Scheme', case=False)
]
table_details_final_noscheme = pd.concat(
    [table_details_final_noscheme, table_details_final_scheme_detail], ignore_index=True)

drop_cols = [c for c in ['Unnamed: 0.1', 'Unnamed: 0'] if c in table_details_final_noscheme.columns]
table_details_final_noscheme.drop(columns=drop_cols, inplace=True)

table_details_final_noscheme['text_new'] = table_details_final_noscheme['text_new'].str.replace('|', '')

# Classify submission status
# Note: website now says "Select to view climate statement" (was "Click to view Climate Statement")
SUBMITTED_PHRASE = r'(click|select) to view climate statement'

table_details_final_noscheme['status'] = np.where(
    table_details_final_noscheme['text_new'].str.contains('no records', case=False), 'No records', '')
table_details_final_noscheme['status'] = np.where(
    table_details_final_noscheme['text_new'].str.contains('Due ', case=False),
    'Have records but not submitted', table_details_final_noscheme['status'])
table_details_final_noscheme['status'] = np.where(
    table_details_final_noscheme['text_new'].str.contains(SUBMITTED_PHRASE, case=False, regex=True),
    'Have records and have submitted', table_details_final_noscheme['status'])

table_details_final_noscheme['scheme'] = np.where(
    table_details_final_noscheme['Scheme Name'].isnull(), 'Company', 'Investment Scheme')

def split_dates(s):
    if ' - Due by ' in str(s):
        a, b = s.split(' - Due by ')
        return a.strip(), b.strip()
    return '0', '0'

table_details_final_noscheme[['Due Date', 'Submit Date']] = (
    table_details_final_noscheme['text_new'].apply(lambda x: pd.Series(split_dates(x))))

print(table_details_final_noscheme['status'].value_counts())
print(table_details_final_noscheme['scheme'].value_counts())
table_details_final_noscheme.to_csv(
    os.path.join(FOLDER, 'pdfs_2026', 'table_details_final_noscheme.final.csv'), index=False)

In [31]:
table_details_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'table_details_final_noscheme.final.csv'))

def extract_period_year(text):
    m = re.search(r'\b\d{2}-\d{2}-(\d{4})\b', str(text))
    return int(m.group(1)) if m else None

table_details_final['period_year'] = table_details_final['text_new'].apply(extract_period_year)

submitted_all = table_details_final[table_details_final['status'] == 'Have records and have submitted']
print(f"Total submitted disclosures (all years): {len(submitted_all)}")
print(submitted_all['period_year'].value_counts().sort_index())

disclose_rows = submitted_all[submitted_all['period_year'].isin(TARGET_YEARS)].copy()
print(f"\nDisclosures available for {TARGET_YEARS}: {len(disclose_rows)}")
print(disclose_rows[['Company Name', 'text_new', 'period_year']].to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\QuyenN\\OneDrive - ESNZ\\Offline_work\\2_GNS\\19_LLM_ClimateRisk2026\\3-Webscrapping & PDF disclosure\\pdfs_2026\\table_details_final_noscheme.final.csv'

In [30]:
# ── Helper functions ───────────────────────────────────────────────────────────

def get_view_details_url(driver, entity_url, target_years):
    """
    Navigate to an entity page and return the 'View details' href for each
    submitted period whose year is in target_years.
    Returns a list of (period_text, detail_url) tuples.
    """
    driver.get(entity_url)
    time.sleep(2)
    results = []
    try:
        table = WebDriverWait(driver, 15).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
        )
        rows = table.find_elements(By.TAG_NAME, "tr")
        for row in rows[1:]:
            cells = row.find_elements(By.TAG_NAME, "td")
            if len(cells) < 2:
                continue
            period_text = cells[0].text
            m = re.search(r'\d{2}-\d{2}-(\d{4})', period_text)
            # Match both old ("Click to view") and new ("Select to view") website text
            is_submitted = bool(re.search(r'(click|select) to view', period_text, re.IGNORECASE))
            if m and int(m.group(1)) in target_years and is_submitted:
                try:
                    a = cells[-1].find_element(By.TAG_NAME, "a")
                    results.append((period_text, a.get_attribute("href")))
                except Exception:
                    pass
    except Exception as e:
        print(f"  Error on {entity_url}: {e}")
    return results


def get_pdf_links(driver, detail_url):
    """
    Navigate to a disclosure detail page and return a list of
    {'text': ..., 'url': ...} dicts for every PDF-like link found.
    """
    driver.get(detail_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    pdf_links = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if any(kw in href.lower() for kw in ['.pdf', '/download', '/document', '/file', '/report']):
            full_url = BASE_URL + href if href.startswith('/') else href
            pdf_links.append({'text': a.get_text(strip=True), 'url': full_url})
    return pdf_links


def download_file(url, save_path, cookies):
    """Download a file to save_path using requests, passing Selenium cookies."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    jar = {c['name']: c['value'] for c in cookies}
    try:
        r = requests.get(url, headers=headers, cookies=jar,
                         stream=True, timeout=60, allow_redirects=True)
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        print(f"    Download error: {e}")
        return False


# ── Main download loop ─────────────────────────────────────────────────────────
download_log = []

print(f"Downloading PDFs for {len(disclose_rows)} disclosures ({TARGET_YEARS})...\n")

for _, row in disclose_rows.iterrows():
    company    = row['Company Name']
    entity_url = row['Link']
    year       = row['period_year']

    print(f"{company} ({year})")

    period_detail_pairs = get_view_details_url(driver, entity_url, TARGET_YEARS)

    if not period_detail_pairs:
        print("  No 'View details' link found")
        download_log.append({'company': company, 'year': year,
                             'entity_url': entity_url, 'status': 'no_detail_link'})
        continue

    for period_text, detail_url in period_detail_pairs:
        pdf_links = get_pdf_links(driver, detail_url)

        if not pdf_links:
            print(f"  No PDFs found on {detail_url}")
            download_log.append({'company': company, 'year': year,
                                 'detail_url': detail_url, 'status': 'no_pdfs'})
            continue

        cookies = driver.get_cookies()
        for i, pdf in enumerate(pdf_links):
            safe_name = re.sub(r'[^\w\s-]', '', company)[:50].strip()
            filename  = f"{safe_name}_{year}_doc{i+1:02d}.pdf"
            save_path = os.path.join(PDF_FOLDER, filename)

            if os.path.exists(save_path):
                print(f"  Already exists: {filename}")
                download_log.append({'company': company, 'year': year, 'url': pdf['url'],
                                     'file': filename, 'status': 'skipped'})
                continue

            print(f"  Downloading: {pdf['text'][:60]} → {filename}")
            ok = download_file(pdf['url'], save_path, cookies)
            download_log.append({'company': company, 'year': year, 'url': pdf['url'],
                                 'file': filename, 'status': 'success' if ok else 'failed'})

log_df = pd.DataFrame(download_log)
log_df.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'download_log_2024_2025.csv'), index=False)
print(f"\n{'='*60}")
print("Download complete.")
print(log_df['status'].value_counts().to_string())

NameError: name 'disclose_rows' is not defined

In [ ]:
driver.quit()
print("Browser closed.")
print(f"\nPDFs saved to : {PDF_FOLDER}")
print(f"Download log  : {os.path.join(FOLDER, 'download_log_2024_2025.csv')}")